![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 3 -- Lab 2: Classification & Cross-Entropy

A hospital has collected tumor measurements from 569 patients. Each tumor is either malignant or benign. The oncology department wants a model that predicts the diagnosis from cell measurements so that biopsies can be prioritized.

You will build a logistic regression classifier from scratch -- implementing the sigmoid function, binary cross-entropy loss, and gradient descent. By the end, you will understand why cross-entropy is the right loss for classification and why MSE falls short.

**Dataset:** Breast Cancer Wisconsin (569 records, 30 features, binary target: malignant vs benign). Loaded via `sklearn.datasets.load_breast_cancer()`.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')

---
## Data Loading

In [ ]:
data = load_breast_cancer()
X_all = data.data
y_all = data.target   # 1 = benign, 0 = malignant
feature_names = data.feature_names
print(f"Shape: {X_all.shape}")
print(f"Classes: {dict(zip(*np.unique(y_all, return_counts=True)))}")
print(f"Features: {list(feature_names[:5])} ... ({len(feature_names)} total)")

---
## Standardization + Train/Test Split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_scaled, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# For Tasks 1-8 we use only the 2 most discriminative features for clean 2D visualization
best_idx = [20, 27]  # worst radius, worst concave points
X_train_2d = X_train_full[:, best_idx]
X_test_2d = X_test_full[:, best_idx]
feat_x, feat_y = feature_names[best_idx[0]], feature_names[best_idx[1]]
print(f"2D features: {feat_x}, {feat_y}")
print(f"Train: {X_train_2d.shape}, Test: {X_test_2d.shape}")

---
## Task 1: Visualize the 2-Feature Space

Plot the training data in the 2D feature space, coloring points by class. This will help you see if the two classes are separable.

**TODO:** Create a scatter plot with feature 1 on x-axis and feature 2 on y-axis. Color malignant (0) and benign (1) points differently. Add a legend.

In [ ]:
# Your code here

---
## Task 2: The Sigmoid Function

The sigmoid function σ(z) = 1 / (1 + e^(-z)) squashes any real number into the range (0, 1), giving us a probability.

**TODO:** Implement `sigmoid(z)` using NumPy. Plot it for z in [-10, 10]. Mark the 0.5 threshold with a horizontal dashed line. In a comment, explain what happens when z → +∞ and z → -∞.

In [ ]:
# Your code here

---
## Task 3: Implement Binary Cross-Entropy

Binary cross-entropy (BCE) measures how far predicted probabilities are from the true labels:

BCE = -(1/n) Σ [y·log(p) + (1-y)·log(1-p)]

It heavily penalizes confident wrong predictions (e.g., predicting 0.99 when truth is 0).

**TODO:** Implement `bce_loss(y_true, y_pred_prob)`. Add a small epsilon (1e-15) inside the log to avoid log(0). Test: if y_true=[1,0,1] and probs=[0.9,0.1,0.8], BCE should be approximately 0.1446.

In [ ]:
# Your code here

---
## Task 4: BCE Penalty Visualization

See exactly how BCE penalizes predictions. For a true label of 1, the loss is -log(p) which explodes as p→0. For a true label of 0, the loss is -log(1-p) which explodes as p→1.

**TODO:** Plot two curves on the same axes: loss vs predicted probability when y=1, and loss vs predicted probability when y=0. Use p from 0.01 to 0.99. This shows why cross-entropy is 'harsh' on confident wrong answers.

In [ ]:
# Your code here

---
## Task 5: Forward Pass

Logistic regression computes: z = Xw + b, then p = σ(z). The prediction is 1 if p ≥ 0.5, else 0.

**TODO:** Write a function `predict_proba(X, w, b)` that returns probabilities using your sigmoid. Initialize w as zeros (shape matching number of features) and b=0. Compute probabilities on X_train_2d and print the first 10 predictions alongside true labels.

In [ ]:
# Your code here

---
## Task 6: Compute Gradients

For logistic regression with BCE loss, the gradients are elegantly simple:

∂L/∂w = (1/n) X^T (p - y)

∂L/∂b = (1/n) Σ (p - y)

where p = σ(Xw + b).

**TODO:** Write a function `compute_gradients(X, y, w, b)` that returns (dw, db). Test at (w=[0,0], b=0).

In [ ]:
# Your code here

---
## Task 7: Training Loop

Put it all together: gradient descent for logistic regression.

**TODO:** Initialize w=[0,0], b=0. Set learning_rate=0.5 and run 300 iterations. Record the BCE loss at each step. Print loss every 50 steps. Plot the loss curve.

In [ ]:
# Your code here

---
## Task 8: Decision Boundary

The decision boundary is where p = 0.5, which means z = w₁x₁ + w₂x₂ + b = 0. Rearranging: x₂ = -(w₁/w₂)x₁ - b/w₂.

**TODO:** Plot the training data colored by class (as in Task 1). Overlay the decision boundary line using your learned w and b. Compute and print the training accuracy.

In [ ]:
# Your code here

---
## Task 9: Compare with sklearn

Now use all 30 features and sklearn's LogisticRegression to see how a production-grade implementation performs.

**TODO:** Fit `LogisticRegression(max_iter=1000)` on `X_train_full`, `y_train`. Predict on `X_test_full`. Print the classification report and plot a confusion matrix heatmap.

In [ ]:
# Your code here

---
## Task 10: Why Not MSE for Classification?

What if we used MSE instead of BCE as the loss for classification? Let's find out.

**TODO:** Implement a training loop identical to Task 7, but replace `bce_loss` with MSE (treating labels 0/1 as numbers). Use the same learning rate and iterations. Compare: (a) final loss values, (b) final training accuracy, (c) loss curve shape. Plot both loss curves on the same axes. In a comment, explain why BCE works better.

In [ ]:
# Your code here

---
## Conclusion

You have built logistic regression from scratch and seen why **binary cross-entropy** is the natural loss for classification:

- The **sigmoid** function converts raw scores into probabilities.
- **BCE** penalizes confident wrong predictions exponentially, driving fast learning.
- **MSE** suffers from vanishing gradients when used with sigmoid, making training sluggish.
- A simple **decision boundary** emerges from the learned weights.
- With all 30 features and sklearn, the model achieves near-perfect accuracy.

In the next lab, you will tackle a challenge that requires choosing between regression and classification -- and picking the right loss function for each.